In [ ]:
import sys
import os
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from sklearn.model_selection import GroupKFold,train_test_split, StratifiedGroupKFold
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from xgboost import XGBClassifier
from sklearn.pipeline import make_pipeline
from sklearn.metrics import confusion_matrix, matthews_corrcoef
from tqdm.notebook import tqdm
import matplotlib.pyplot as plt
import seaborn as sns
from mantis.architecture import MantisV2
from mantis.trainer import MantisTrainer

# 強制指向你的專案根目錄 (Mantis 資料夾的上一層)
project_root = r"F:\M143020071\MI\program"
if project_root not in sys.path:
    sys.path.append(project_root)

# 使用別名 mantis_tool，後面呼叫比較方便
from Mantis import single_channel_extract_feats as mantis_tool

In [ ]:
def get_metrics(y_true, y_pred):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0,1]).ravel()
    denom = tp + tn + fp + fn
    acc = (tp + tn) / denom if denom > 0 else 0
    sens = tp / (tp + fn) if (tp + fn) > 0 else 0
    spec = tn / (tn + fp) if (tn + fp) > 0 else 0
    mcc = matthews_corrcoef(y_true, y_pred)
    return [acc, sens, spec, mcc]

def evaluate_predictions(y_true, y_prob, groups, threshold=0.5):
    """統一處理 Window-level 與 Subject-level 的指標計算"""
    win_metrics = get_metrics(y_true, (y_prob >= threshold).astype(int))
    
    # 計算 Subject-level (以受試者平均機率判定)
    df = pd.DataFrame({'ID': groups, 'Actual': y_true, 'Prob': y_prob}).groupby('ID').mean()
    sub_metrics = get_metrics(df['Actual'], (df['Prob'] >= threshold).astype(int))
    
    return win_metrics, sub_metrics, df

def load_data_with_groups(MI_path, non_MI_path):
    X, y, groups = [], [], []
    for path, label in [(MI_path, 1), (non_MI_path, 0)]:
        if not os.path.exists(path): continue
        for file in [f for f in os.listdir(path) if f.endswith('.npy')]:
            data = np.load(os.path.join(path, file))[:, 1:].astype(np.float32)  # 假設第一欄是時間戳，丟掉
            X.append(data)
            y.append(np.full(len(data), label))
            groups.extend([file] * len(data))
    return np.vstack(X), np.concatenate(y), np.array(groups)


def plot_confusion_matrix(y_true, y_pred, title, ax):
    cm = confusion_matrix(y_true, y_pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
                xticklabels=['Non-MI(0)', 'MI(1)'], 
                yticklabels=['Non-MI(0)', 'MI(1)'], ax=ax)
    ax.set_title(title)
    ax.set_ylabel('Actual')
    ax.set_xlabel('Predicted')

def save_cm_to_csv(y_true, y_pred, filename):
    """計算混淆矩陣並存成 CSV"""
    cm = confusion_matrix(y_true, y_pred)
    # 建立 DataFrame 並加入行列標籤增加可讀性
    cm_df = pd.DataFrame(cm, 
                         index=['Actual_0', 'Actual_1'], 
                         columns=['Predicted_0', 'Predicted_1'])
    full_path = os.path.join(save_path, f"{filename}.csv")
    cm_df.to_csv(full_path)
    print(f"Saved: {full_path}")

In [ ]:
base_path = r"D:\M143020071\MI\raw_data_result\iSKNA_signal\ch1\sr10000_500_3000_MI_1000pts_win20s_step2s"
save_path = r"D:\M143020071\MI\xgboost_results\iSKNA_signal\ch1\sr10000_500_3000_MI_1000pts_win20s_step2s\mantisPCA_test"

X_raw, y, groups = load_data_with_groups(os.path.join(base_path, "MI"), os.path.join(base_path, "non_MI"))
print(X_raw.shape)
if not os.path.exists(save_path):
    os.makedirs(save_path)
X_raw_resized = mantis_tool.resize(X_raw)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
network = MantisV2(device=device, return_transf_layer=0, output_token='combined')
network = network.from_pretrained("paris-noah/MantisV2")
model_trainer = MantisTrainer(device=device, network=network)

# X_raw_transformed = model_trainer.transform(X_raw_resized)
# print(X_raw_resized.shape, X_raw_transformed.shape)


In [ ]:
for layer_idx in range(6):
    print(f"正在處理第 {layer_idx+1} 層...")
    network.return_transf_layer = layer_idx
    transformed_data = model_trainer.transform(X_raw_resized)
    print(f"第 {layer_idx+1} 層轉換完成，形狀: {transformed_data.shape}")
    file_name = f"X_raw_transformed_{layer_idx}.npy"
    full_save_path = os.path.join(save_path, file_name)
    np.save(full_save_path, transformed_data)
    

print("所有層級轉換完成並已儲存。")

In [ ]:
hyper_params_dict = {
    'objective': 'binary:logistic',
    'booster': 'gbtree', 
    'eval_metric': 'aucpr', 
    'learning_rate': 0.05, 
    'n_estimators': 500, 
    'max_depth': 3, 
    'min_child_weight': 1,
    'gamma': 0.1, 
    'subsample': 0.8, 
    'colsample_bytree': 0.8, 
    'reg_alpha': 0.01, 
    'reg_lambda': 1, 
    'random_state': 42
}

batch1 = [3, 21, 34, 51, 61, 65, 72, 74, 78, 84, 99, 112, 114, 127, 147, 149, 154, 168, 200, 201, 212, 220, 234, 235, 244, 256, 257, 264, 265, 270, 288, 329, 331, 337, 345, 354, 364, 383, 406, 407, 427, 432, 467, 482, 483, 509, 510, 515, 516, 529, 554, 556, 558, 588, 597, 610, 616, 624, 625, 628, 629, 633, 637, 646, 657, 673, 686, 688, 701, 721, 751, 757, 776, 785, 797, 835, 858, 867, 895, 903]
batch2 = [14, 15, 26, 38, 55, 58, 86, 92, 94, 105, 110, 130, 134, 135, 140, 141, 144, 145, 155, 169, 183, 193, 211, 226, 238, 239, 249, 274, 280, 285, 305, 313, 333, 339, 348, 367, 386, 404, 414, 419, 444, 447, 448, 452, 458, 464, 472, 473, 477, 486, 487, 500, 506, 537, 560, 574, 582, 598, 600, 614, 650, 665, 668, 669, 681, 705, 711, 725, 748, 791, 801, 816, 818, 821, 827, 830, 838, 871, 892, 900]
batch3 = [11, 13, 18, 29, 32, 35, 47, 49, 60, 73, 77, 82, 89, 104, 113, 115, 128, 151, 197, 203, 213, 214, 228, 237, 240, 271, 294, 302, 312, 338, 341, 346, 361, 362, 368, 372, 384, 394, 399, 403, 425, 442, 463, 470, 475, 494, 522, 525, 532, 541, 544, 553, 559, 563, 584, 587, 593, 595, 607, 620, 640, 654, 682, 697, 706, 723, 750, 760, 762, 763, 772, 820, 834, 883, 885, 886, 891, 901, 909, 921]
batch4 = [7, 12, 16, 20, 36, 62, 64, 71, 108, 123, 146, 152, 156, 191, 198, 225, 243, 251, 262, 290, 301, 304, 320, 325, 328, 332, 349, 351, 358, 365, 380, 391, 396, 397, 408, 409, 416, 421, 422, 424, 433, 436, 459, 465, 468, 479, 481, 489, 495, 496, 511, 520, 531, 538, 566, 575, 592, 618, 627, 632, 638, 639, 642, 655, 664, 667, 683, 684, 696, 710, 719, 731, 794, 807, 853, 863, 874, 879, 880, 893]
batch5 = [2, 10, 22, 24, 48, 87, 103, 119, 124, 157, 159, 160, 167, 178, 209, 210, 216, 219, 221, 233, 268, 269, 291, 295, 299, 300, 303, 308, 336, 342, 344, 366, 369, 371, 375, 378, 410, 412, 415, 417, 445, 446, 449, 451, 453, 455, 499, 508, 512, 528, 543, 552, 562, 578, 605, 606, 608, 611, 626, 656, 661, 671, 692, 702, 736, 739, 740, 768, 769, 793, 802, 809, 810, 823, 839, 842, 866, 894, 899, 910]

all_batches = [batch1, batch2, batch3, batch4, batch5]

gkf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)
all_layer_results = []


for layer_idx in range(6):
    print(f"\n========== 正在處理 MantisV2 第 {layer_idx+1} 層 ==========")
    file_name = os.path.join(save_path, f"X_raw_transformed_{layer_idx}.npy")
    X_raw_transformed=np.load(file_name)
    print(f"已載入特徵檔案: {file_name} | 形狀: {X_raw_transformed.shape}")
    fold_results = []
    layer_y_true = []
    layer_y_probs = []
    layer_groups = []
  
    # for fold, (train_idx, test_idx) in enumerate(gkf.split(X_raw, y, groups)):
    for fold, test_batch in enumerate(all_batches):
        print(f"  --> 執行 Fold {fold + 1}/5")
        
        # X_train, X_test = X_raw_transformed[train_idx], X_raw_transformed[test_idx]
        # y_train, y_test = y[train_idx], y[test_idx]
        # groups_test = groups[test_idx] # evaluate_predictions
        test_group_names = [f"{i:04d}.npy" for i in test_batch]
        is_test = np.isin(groups, test_group_names)
        test_idx = np.where(is_test)[0]
        train_idx = np.where(~is_test)[0]
        
        
        X_train, X_test = X_raw_transformed[train_idx], X_raw_transformed[test_idx]
        y_train, y_test = y[train_idx], y[test_idx]
        groups_test = groups[test_idx]
        
        

        pipeline = make_pipeline(
            StandardScaler(),
            PCA(n_components=0.95, random_state=42), 
            XGBClassifier(**hyper_params_dict)
        )
        pipeline.fit(X_train, y_train)
        test_probs = pipeline.predict_proba(X_test)[:, 1]

        layer_y_true.append(y_test)
        layer_y_probs.append(test_probs)
        layer_groups.append(groups_test)
        
        
      
    y_true_combined = np.concatenate(layer_y_true)
    y_probs_combined = np.concatenate(layer_y_probs)
    groups_combined = np.concatenate(layer_groups)

    win_metrics, sub_metrics, df_sub = evaluate_predictions(y_true_combined, y_probs_combined, groups_combined)
    all_layer_results.append({
        'Layer': layer_idx + 1,
        'Win_Acc': win_metrics[0], 'Win_Sens': win_metrics[1], 'Win_Spec': win_metrics[2], 'Win_MCC': win_metrics[3],
        'Sub_Acc': sub_metrics[0], 'Sub_Sens': sub_metrics[1], 'Sub_Spec': sub_metrics[2], 'Sub_MCC': sub_metrics[3]
    })

    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    plot_confusion_matrix(y_true_combined, (y_probs_combined >= 0.5).astype(int), 
                          f"Layer {layer_idx+1}: Cumulative Window-level", axes[0])
    plot_confusion_matrix(df_sub['Actual'], (df_sub['Prob'] >= 0.5).astype(int), 
                          f"Layer {layer_idx+1}: Cumulative Subject-level", axes[1])
    plt.tight_layout()
    plt.savefig(os.path.join(save_path, f"cumulative_cm_layer_{layer_idx+1}.png"))
    
    save_cm_to_csv(y_true_combined, (y_probs_combined >= 0.5).astype(int), f"cm_win_cumulative_layer_{layer_idx+1}")
    save_cm_to_csv(df_sub['Actual'], (df_sub['Prob'] >= 0.5).astype(int), f"cm_sub_cumulative_layer_{layer_idx+1}")
    
    plt.show()

summary_df = pd.DataFrame(all_layer_results)
display(summary_df)
summary_df.to_csv(os.path.join(save_path, "Xgboost_MantisPCA_5Fold_Results0423.csv"), index=False)